### Umgang mit den Wetterdaten 

Bestehende wissenschaftliche Ansätze zur Platzierungsvorhersage im Straßenradsport (wie z. B. Kholkine et al., *A Learn-to-Rank Approach for Predicting Road Cycling Race Outcomes*) erwähnen meteorologische Einflüsse zwar als theoretisch relevant, klammern komplexe Faktoren wie den Wind in ihren Modellen jedoch meist vollständig aus. Der Grund hierfür liegt in der methodischen Schwierigkeit, ohne hochauflösende GPS-Trajektorien (GPX-Daten) präzise physikalische Vektoren für die kurvenreichen Rennstrecken zu berechnen. 
Da wir uns auf das Ranking fokussieren und keine Vorraussagung einer möglichen Fluchtgruppe machen, scheint uns dieser Ansatz passend.

Im Rahmen dieser Arbeit versuchen wir die Wetterdaten anhand einer Abstraktion zu modellieren. Wir modellieren den Wind nicht als direkten, kleinteiligen Richtungsvektor, sondern als "Hintergrund-Faktor" (Contextual Feature).

Hierzu wurde ein *Wind-Stabilitäts-Index* entwickelt. Dieser misst die absolute Varianz der Windrichtung und Windgeschwindigkeit zwischen dem Start- und Zielort. 

* **Die wissenschaftliche Logik:** Ein hoher Index signalisiert dem Modell eine instabile, stark drehende Wetterlage über den Tagesverlauf. Dies erhöht die allgemeine Dynamik und das taktische Risiko für Feldteilungen (Windkanten), ohne das Modell mit pseudo-präzisem Rauschen zu überfüttern. Das Feature fungiert somit als sanfter Varianz-Indikator, während die primären sportlichen Metriken (wie das Saisonranking) die Hauptlast der Platzierungsvorhersage tragen.

In [1]:
import pandas as pd
import os
import numpy as np

In [2]:
df = pd.read_pickle('../../data/processed/20_cleaned_master_data.pkl')

In [3]:
wind_spalten = [
    'departure_wind_mittel',
    'arrival_wind_mittel',
    'departure_windrichtung_mittel',
    'arrival_windrichtung_mittel'
]

print(df[wind_spalten].head(20).to_string())


   departure_wind_mittel arrival_wind_mittel departure_windrichtung_mittel arrival_windrichtung_mittel
0                  12.07                6.22                        328.25                      277.25
1                  12.07                6.22                        328.25                      277.25
2                  12.07                6.22                        328.25                      277.25
3                  12.07                6.22                        328.25                      277.25
4                  12.07                6.22                        328.25                      277.25
5                  12.07                6.22                        328.25                      277.25
6                  12.07                6.22                        328.25                      277.25
7                  12.07                6.22                        328.25                      277.25
8                  12.07                6.22                        328.2

In [4]:
# 1. Deltas berechnen
df['wind_speed_delta'] = np.abs(df['departure_wind_mittel'] - df['arrival_wind_mittel'])

richtung_diff = np.abs(df['departure_windrichtung_mittel'] - df['arrival_windrichtung_mittel']) % 360
df['wind_direction_delta'] = np.where(richtung_diff > 180, 360 - richtung_diff, richtung_diff)

# 2. Maximum für die relative Skalierung bestimmen
max_speed_delta = df['wind_speed_delta'].max()
if max_speed_delta == 0:
    max_speed_delta = 1

# 3. Den sanften Index berechnen (0 = maximal stabil, 1 = maximal unruhig)
# Windgeschwindigkeit und Richtung jeweils 50% Einfluss
df['wind_stability_index'] = (
    (df['wind_speed_delta'] / max_speed_delta) * 0.5 +
    (df['wind_direction_delta'] / 180.0) * 0.5
)

In [5]:
wind_spalten = [
    'departure_wind_mittel',
    'arrival_wind_mittel',
    'departure_windrichtung_mittel',
    'arrival_windrichtung_mittel',
    'wind_speed_delta',
    'wind_direction_delta',
    'wind_stability_index'
]

print(df[wind_spalten].head(20).to_string())

   departure_wind_mittel arrival_wind_mittel departure_windrichtung_mittel arrival_windrichtung_mittel wind_speed_delta wind_direction_delta wind_stability_index
0                  12.07                6.22                        328.25                      277.25             5.85                 51.0             0.238106
1                  12.07                6.22                        328.25                      277.25             5.85                 51.0             0.238106
2                  12.07                6.22                        328.25                      277.25             5.85                 51.0             0.238106
3                  12.07                6.22                        328.25                      277.25             5.85                 51.0             0.238106
4                  12.07                6.22                        328.25                      277.25             5.85                 51.0             0.238106
5                  12.07    

### Weitere Wetterdaten
- Start und Zielwerte sind meist recht ähnlich
- Somit entstehen viele hochkorrelierte Spalten (Multikollinearität) -> dies kann dazu führen, dass das Modell "verwirrt" wird und die Wichtigkeit der einzelnen Features (Feature Importance) nicht mehr sauber zuweisen kann.
- Somit bringen wir auch die restlichen Werte in eine smarte Hybrid-Feature-Struktur, indem wir das allgemeine Niveau (Mittelwert) und die meteorologische Tendenz (Trend) voneinander entkoppeln.

In [6]:
wetter_roh_spalten = [
    'departure_temp_mittel', 'departure_regen_mittel', 'departure_wind_mittel',
    'departure_luftfeuchte_mittel', 'departure_niederschlag_mittel', 'departure_windrichtung_mittel',
    'arrival_temp_mittel', 'arrival_regen_mittel', 'arrival_wind_mittel',
    'arrival_luftfeuchte_mittel', 'arrival_niederschlag_mittel', 'arrival_windrichtung_mittel'
]
print(df[wetter_roh_spalten].head(5).to_string())

  departure_temp_mittel departure_regen_mittel departure_wind_mittel departure_luftfeuchte_mittel departure_niederschlag_mittel departure_windrichtung_mittel arrival_temp_mittel arrival_regen_mittel arrival_wind_mittel arrival_luftfeuchte_mittel arrival_niederschlag_mittel arrival_windrichtung_mittel
0                 23.82                    0.0                 12.07                         69.0                           0.0                        328.25               28.32                 0.05                6.22                       50.0                        0.05                      277.25
1                 23.82                    0.0                 12.07                         69.0                           0.0                        328.25               28.32                 0.05                6.22                       50.0                        0.05                      277.25
2                 23.82                    0.0                 12.07                         6

In [7]:
# 1. TEMPERATUR: Allgemeines Niveau & echter Trend (Ziel - Start)
df['weather_temp_mean'] = (df['departure_temp_mittel'] + df['arrival_temp_mittel']) / 2
df['weather_temp_trend'] = df['arrival_temp_mittel'] - df['departure_temp_mittel']

# 2. REGEN: Mittlere Wahrscheinlichkeit
df['weather_rain_prob_mean'] = (df['departure_regen_mittel'] + df['arrival_regen_mittel']) / 2

# 3. NIEDERSCHLAG: Mittlere Menge
df['weather_precipitation_mean'] = (df['departure_niederschlag_mittel'] + df['arrival_niederschlag_mittel']) / 2

# 4. LUFTFEUCHTIGKEIT: Mittleres Niveau
df['weather_humidity_mean'] = (df['departure_luftfeuchte_mittel'] + df['arrival_luftfeuchte_mittel']) / 2

In [8]:
wetter_neu = ['url', 'weather_temp_mean', 'weather_temp_trend', 'weather_rain_prob_mean', 'weather_humidity_mean']
print(df[wetter_neu].drop_duplicates().head(5).to_string(index=False))

                                                             url weather_temp_mean weather_temp_trend weather_rain_prob_mean weather_humidity_mean
https://www.procyclingstats.com/race/tour-de-france/2005/stage-2             26.07                4.5                  0.025                  59.5
https://www.procyclingstats.com/race/tour-de-france/2005/stage-3             20.27                1.5                    0.0                  39.0
https://www.procyclingstats.com/race/tour-de-france/2005/stage-5             19.14               2.68                  0.105                  57.5
https://www.procyclingstats.com/race/tour-de-france/2005/stage-6            15.975              -1.65                   1.05                73.625
https://www.procyclingstats.com/race/tour-de-france/2005/stage-7             16.82                1.5                   0.25                  73.5


In [9]:
pfad = '../../data/processed/21_cleaned_master_data.pkl'
df.to_pickle(pfad)